# Семинар 04. Классы, наследование и полиморфизм


## Цели

После семинара вы сможете:

- создавать классы с атрибутами экземпляра и класса;
- использовать наследование и переопределение методов;
- выбирать видимость атрибутов и общий интерфейс объектов.

## Перед началом

Повторите функции, параметры, возвращаемые значения и материал семинара 03.


## Когда нужен класс

Класс связывает состояние объекта с операциями, которые обязаны сохранять правила этого состояния. Если данные и функции живут независимо, класс не нужен: оборачивать каждую пару функций в `Manager`, `Service` или `Helper` — не ООП, а лишний уровень имён.

Основные идеи:

- **Абстракция:** клиент работает с понятным контрактом и не обязан знать устройство реализации.
- **Инкапсуляция:** объект сам отвечает за своё допустимое состояние; внутренние детали не становятся частью публичного контракта без причины.
- **Наследование:** подкласс повторно использует и уточняет контракт базового класса. Это отношение «является», а не универсальный механизм экономии строк кода.
- **Полиморфизм:** один клиентский код работает с разными объектами через общий контракт. В Python для этого часто достаточно duck typing; наследование не обязательно.

## Экземпляр, класс и метод

Экземпляр хранит собственное состояние. Класс описывает поведение экземпляров и может хранить общие атрибуты. При вызове `pet.say()` Python находит функцию `say` в классе и передаёт `pet` первым аргументом `self`. `self` не ключевое слово, а обязательное соглашение, без которого код быстро станет непонятным.

Состояние лучше менять через операции предметной области, которые проверяют инварианты. Бессмысленные пары `get_x()`/`set_x()` для каждого поля ничего не инкапсулируют. Если при присваивании нужна проверка, в Python обычно используют `property`.


In [ ]:
class Animal:
    def __init__(self, name: str) -> None:
        self.name = name

    @property
    def name(self) -> str:
        return self._name

    @name.setter
    def name(self, value: str) -> None:
        value = value.strip()
        if not value:
            raise ValueError("Name must not be empty")
        self._name = value

    def __str__(self) -> str:
        return f"Animal named {self.name}"

    def say(self) -> None:
        print("Animals can't talk")

pet = Animal("Lucky")
pet2 = Animal("Not so Lucky")
print(pet)
pet.say()
pet.name = "Marquise"
print(pet)
pet2.say()

## Атрибуты экземпляра и класса

При чтении `obj.attribute` Python сначала ищет атрибут у экземпляра, затем в его классе и базовых классах. Присваивание `obj.legs = 3` создаёт атрибут экземпляра и заслоняет одноимённый атрибут класса; оно не меняет класс.

Изменяемый объект вроде списка в атрибуте класса будет общим для всех экземпляров. Это частый источник ошибок: индивидуальные списки и словари нужно создавать в `__init__`.

- Обычный метод получает экземпляр `self`.
- `classmethod` получает класс `cls` и подходит для альтернативных конструкторов.
- `staticmethod` не получает ни экземпляр, ни класс. Это обычная функция, помещённая в пространство имён класса; если она не относится к классу по смыслу, ей место на уровне модуля.


In [ ]:
class Animal:
    legs: int = 4

    def __init__(self, name: str) -> None:
        if not self.is_valid_name(name):
            raise ValueError("Name must not be empty")
        self.name = name

    def __str__(self) -> str:
        return f"Animal named {self.name}, legs: {self.legs}"

    @classmethod
    def from_description(cls, description: str) -> "Animal":
        _, name = description.split(":", maxsplit=1)
        return cls(name.strip())

    @staticmethod
    def is_valid_name(name: str) -> bool:
        return bool(name.strip())


class Dog(Animal):
    pass


pet = Animal("Lucky")
another_pet = Animal.from_description("name: Marquise")
dog = Dog.from_description("name: Rex")
assert isinstance(dog, Dog)

# Атрибут экземпляра заслоняет атрибут Animal.legs только для pet.
pet.legs = 3
print(pet)
print(another_pet)

# Изменение класса видно экземплярам без собственного атрибута legs.
Animal.legs = 5
print(another_pet)
print(pet)

## Наследование и полиморфизм

Наследование уместно, когда подкласс действительно выполняет контракт базового класса и может использоваться вместо него. Подкласс может использовать, переопределять и добавлять методы, но обязан сохранять смысл унаследованных операций. Иерархия, построенная только ради повторного использования пары методов, обычно хуже композиции.

Полиморфизм виден в функции, которая вызывает один и тот же метод у разных объектов и не содержит цепочку `if type(...)`. Если для каждого нового животного приходится добавлять ещё одну ветку, общий контракт не работает.

## Соглашения о видимости

- `name` — публичная часть интерфейса.
- `_name` — внутренний атрибут по соглашению. Доступ технически открыт, но внешний код не должен на него опираться.
- `__name` запускает name mangling: внутри `Animal` имя станет `_Animal__name`. Это защищает прежде всего от случайного конфликта имён в наследнике, а не от доступа. Настоящего модификатора private в Python нет.

Подчёркивание — сигнал клиенту, а не забор с охраной. Внешний код может нарушить соглашение, но тогда сам подписывается на поломку при следующем рефакторинге.

## Композиция

Наследование выражает отношение «является», композиция — «содержит» или «использует». Автомобиль использует двигатель, но не является двигателем. Если часть поведения должна заменяться независимо, композиция обычно даёт более честную модель:

```python
class Engine:
    def start(self) -> None:
        print("Engine started")


class Car:
    def __init__(self, engine: Engine) -> None:
        self._engine = engine

    def start(self) -> None:
        self._engine.start()
```

## Множественное наследование и MRO

При нескольких базовых классах Python строит порядок разрешения атрибутов — MRO. Его можно увидеть в `Class.__mro__`. Вызов `super()` идёт к следующему классу в MRO, а не обязательно к «родителю, написанному сверху».

Множественное наследование узких интерфейсов и аккуратных mixin-классов нормально. Несколько базовых классов с собственным состоянием, несовместимыми `__init__` и конкурирующими методами — минное поле. Там композиция обычно проще и честнее.


In [ ]:
from abc import ABC, abstractmethod


class Animal(ABC):
    @abstractmethod
    def talk(self) -> None:
        ...


class Dog(Animal):
    def talk(self) -> None:
        print("woof")


class Cat(Animal):
    def talk(self) -> None:
        print("meow")


class PussInBoots(Cat):
    def talk(self) -> None:
        print("Fear me, if you dare!")
        super().talk()


animals = [Dog(), Cat(), PussInBoots()]
for animal in animals:
    animal.talk()


# Dog стоит раньше Cat, поэтому CatDog.talk берётся из Dog.
class CatDog(Dog, Cat):
    pass


CatDog().talk()
print([cls.__name__ for cls in CatDog.__mro__])


class Flyable(ABC):
    @abstractmethod
    def fly(self) -> None:
        ...
    

class Swimmable(ABC):
    @abstractmethod
    def swim(self) -> None:
        ...


class BaseFlyer(Flyable):
    def fly(self) -> None:
        print("I'm flying")


class FlyingFish(BaseFlyer, Swimmable):
    def swim(self) -> None:
        print("I'm swimming deep")


class Duck(Flyable, Swimmable):
    def fly(self) -> None:
        print("I'm flying")

    def swim(self) -> None:
        print("I'm swimming")


fish = FlyingFish()
fish.fly()
fish.swim()


## Самопроверка

1. Когда набор данных и функций действительно стоит превращать в класс?
2. Чем атрибут экземпляра отличается от атрибута класса?
3. Что произойдёт при присваивании `pet.legs = 3`, если `legs` определён только в классе?
4. Чем обычный метод, `classmethod` и `staticmethod` отличаются по неявному первому аргументу?
5. Почему имя с `__` нельзя считать настоящим private-атрибутом?
6. Когда композиция честнее наследования?
7. Куда ведёт `super()` при множественном наследовании?


## Итоги

- Класс полезен, когда состояние и операции образуют единый объект с инвариантами.
- Атрибуты экземпляра индивидуальны, атрибуты класса переиспользуются, пока экземпляр их не заслонит.
- `property` позволяет сохранить обычный синтаксис атрибута и контролировать присваивание.
- Наследование обязано сохранять контракт базового класса; повторное использование кода само по себе его не оправдывает.
- Полиморфизм убирает проверки конкретных типов из клиентского кода.
- При множественном наследовании поиск и `super()` следуют MRO.
- Композиция подходит для частей, которые объект содержит, использует или должен независимо заменять.

Полезные источники: [классы и name mangling](https://docs.python.org/3/tutorial/classes.html), [`classmethod` и `staticmethod`](https://docs.python.org/3/library/functions.html#classmethod), [порядок разрешения методов](https://docs.python.org/3/howto/mro.html).


## Задание 1. Класс-шифровальщик (1 балл)
```python
class Cipher:
    def __init__(self, key):
        self.__key = key

    def encrypt(self, src: str) -> str:
        pass

    def decrypt(self, src: str) -> str:
        pass   

cipher = Cipher('q')
cipher.encrypt('python')
```


**Критерии проверки:** ключ хранится в экземпляре; методы не меняют исходную строку; `decrypt(encrypt(text)) == text` для пустой строки, обычного текста и символов вне алфавита.


## Задание 2. Выбор алгоритма шифрования (2 балла)

```python
class BaseCipher(abc.ABC):
    def __init__(self, key):
        self.key = key

    @abc.abstractmethod
    def encrypt(self, src: str) -> str:
        pass

    @abc.abstractmethod
    def decrypt(self, src: str) -> str:
        pass

    @staticmethod
    def create_cipher(algorithm: str, key) -> 'BaseCipher':
        raise NotImplementedError()


class CaesarCipher(BaseCipher):
    pass

class VigenereCipher(BaseCipher):
    pass

cipher = BaseCipher.create_cipher('caesar', 'a')
...
```
**Критерии проверки:** оба класса реализуют общий абстрактный интерфейс; фабрика создаёт нужный класс и отклоняет неизвестный алгоритм с `ValueError`; для обоих алгоритмов выполняется обратимость.
